In [7]:
'''
Takes socialiqa_results_* nameed files from results folder and concatenates them, to creatre a single unified folder of results
Also validates every sample t ensure it has all required  output fields
'''

import json
import os
from pathlib import Path
from collections import defaultdict
from datetime import datetime

RESULTS_DIR = Path.cwd() / "results" # Requires this file to be in metamind_fork, as a neighbor to results
OUTPUT_FILE = RESULTS_DIR / f"all_results_combined_{datetime.now().strftime('%Y%m%d_%H%M%S')}.jsonl"

REQUIRED_FIELDS = {
    "sample_id", "input", "gold_answer", "predicted_answer",
    "correct", "final_response", "tom_hypotheses", "selected_hypothesis",
    "response_details", "elapsed_seconds", "llm_calls_so_far"
}

In [8]:
print(Path.cwd())
files = sorted(RESULTS_DIR.glob("socialiqa_results_*.jsonl"))
print(f"Found {len(files)} result files:")
for f in files:
    print(f"  {f.name}")

/Users/eitan/Documents/School related/Capstone Project/metamind_fork
Found 65 result files:
  socialiqa_results_20260304_111247.jsonl
  socialiqa_results_20260304_111449.jsonl
  socialiqa_results_20260304_122448.jsonl
  socialiqa_results_20260304_123355.jsonl
  socialiqa_results_20260304_224947.jsonl
  socialiqa_results_20260305_000328.jsonl
  socialiqa_results_20260305_001908.jsonl
  socialiqa_results_20260305_125554.jsonl
  socialiqa_results_20260305_130201.jsonl
  socialiqa_results_20260306_002051.jsonl
  socialiqa_results_20260306_002052.jsonl
  socialiqa_results_20260306_002319.jsonl
  socialiqa_results_20260306_005320.jsonl
  socialiqa_results_20260306_101821.jsonl
  socialiqa_results_20260306_101822.jsonl
  socialiqa_results_20260306_101823.jsonl
  socialiqa_results_20260307_142034.jsonl
  socialiqa_results_20260308_000706_s337.jsonl
  socialiqa_results_20260308_000706_s6363.jsonl
  socialiqa_results_20260308_000706_s6847.jsonl
  socialiqa_results_20260308_003215_s7342.jsonl
  s

In [9]:
all_samples = {}
stats = defaultdict(int)
skipped = []

for f in files:
    file_valid, file_skipped = 0, 0
    with open(f) as fh:
        for i, line in enumerate(fh, 1):
            line = line.strip()
            if not line:
                continue
            try:
                d = json.loads(line)
            except json.JSONDecodeError as e:
                skipped.append({"file": f.name, "line": i, "reason": f"JSON error: {e}"})
                file_skipped += 1
                continue

            missing = REQUIRED_FIELDS - d.keys()
            if missing:
                skipped.append({"file": f.name, "line": i, "sample_id": d.get("sample_id", "?"), "reason": f"Missing fields: {missing}"})
                file_skipped += 1
                continue

            sid = d["sample_id"]
            if sid in all_samples:
                stats["duplicates"] += 1
            all_samples[sid] = d
            file_valid += 1

    stats["total_valid"] += file_valid
    stats["total_skipped"] += file_skipped
    print(f"{f.name}: {file_valid} valid, {file_skipped} skipped")

print(f"\nTotal unique samples: {len(all_samples)}")
print(f"Total skipped: {stats['total_skipped']}")
print(f"Duplicates overwritten: {stats['duplicates']}")

socialiqa_results_20260304_111247.jsonl: 150 valid, 0 skipped
socialiqa_results_20260304_111449.jsonl: 149 valid, 0 skipped
socialiqa_results_20260304_122448.jsonl: 152 valid, 0 skipped
socialiqa_results_20260304_123355.jsonl: 148 valid, 0 skipped
socialiqa_results_20260304_224947.jsonl: 186 valid, 0 skipped
socialiqa_results_20260305_000328.jsonl: 178 valid, 0 skipped
socialiqa_results_20260305_001908.jsonl: 173 valid, 2 skipped
socialiqa_results_20260305_125554.jsonl: 110 valid, 11 skipped
socialiqa_results_20260305_130201.jsonl: 177 valid, 0 skipped
socialiqa_results_20260306_002051.jsonl: 182 valid, 0 skipped
socialiqa_results_20260306_002052.jsonl: 176 valid, 0 skipped
socialiqa_results_20260306_002319.jsonl: 176 valid, 0 skipped
socialiqa_results_20260306_005320.jsonl: 178 valid, 0 skipped
socialiqa_results_20260306_101821.jsonl: 147 valid, 36 skipped
socialiqa_results_20260306_101822.jsonl: 181 valid, 0 skipped
socialiqa_results_20260306_101823.jsonl: 168 valid, 0 skipped
social

In [10]:
sorted_samples = [all_samples[k] for k in sorted(all_samples.keys())]

with open(OUTPUT_FILE, "w") as out:
    for s in sorted_samples:
        out.write(json.dumps(s) + "\n")

print(f"Written to {OUTPUT_FILE}")

Written to /Users/eitan/Documents/School related/Capstone Project/metamind_fork/results/all_results_combined_20260317_204751.jsonl


In [11]:
correct = sum(1 for s in sorted_samples if s["correct"])
total = len(sorted_samples)
accuracy = correct / total * 100 if total else 0
avg_elapsed = sum(s["elapsed_seconds"] for s in sorted_samples) / total if total else 0
avg_llm_calls = sum(s["llm_calls_so_far"] / max(s["sample_id"] - sorted_samples[0]["sample_id"] + 1, 1) for s in sorted_samples)

sample_ids = sorted(all_samples.keys())
full_range = set(range(sample_ids[0], sample_ids[-1] + 1))
missing_ids = sorted(full_range - set(sample_ids))

print("=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"Total samples:       {total}")
print(f"Correct:             {correct}")
print(f"Accuracy:            {accuracy:.2f}%")
print(f"Avg elapsed:         {avg_elapsed:.1f}s")
print(f"Sample ID range:     {sample_ids[0]} - {sample_ids[-1]}")
print(f"Missing IDs in range:{len(missing_ids)}")
if missing_ids[:20]:
    print(f"  First 20 missing:  {missing_ids[:20]}")
if skipped:
    print(f"\nSkipped entries ({len(skipped)}):")
    for s in skipped:
        print(f"  {s}")

SUMMARY
Total samples:       9286
Correct:             7457
Accuracy:            80.30%
Avg elapsed:         163.2s
Sample ID range:     1 - 20149
Missing IDs in range:10863
  First 20 missing:  [828, 829, 830, 831, 832, 833, 834, 835, 836, 870, 871, 880, 881, 917, 918, 919, 920, 921, 922, 923]

Skipped entries (49):
  {'file': 'socialiqa_results_20260305_001908.jsonl', 'line': 12, 'reason': "JSON error: Expecting ':' delimiter: line 1 column 5285 (char 5284)"}
  {'file': 'socialiqa_results_20260305_001908.jsonl', 'line': 172, 'reason': 'JSON error: Expecting value: line 1 column 1 (char 0)'}
  {'file': 'socialiqa_results_20260305_125554.jsonl', 'line': 1, 'reason': "JSON error: Expecting ':' delimiter: line 1 column 4924 (char 4923)"}
  {'file': 'socialiqa_results_20260305_125554.jsonl', 'line': 2, 'reason': "JSON error: Expecting ',' delimiter: line 1 column 374 (char 373)"}
  {'file': 'socialiqa_results_20260305_125554.jsonl', 'line': 3, 'reason': "JSON error: Expecting ':' delimite